In [1]:
import sqlite3
import pandas as pd
import numpy as np
import zipfile
import os
from google.colab import files

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(".")

db_files = [f for f in os.listdir('.') if f.endswith('.db')]
db_path = db_files[0]

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
table_name = cursor.fetchall()[0][0]

query = f'''
SELECT id, title, price, area, beds, livings, wc, ac, ketchen, furnished, district, create_time
FROM {table_name}
WHERE city = 'الرياض'
  AND category = 1
  AND rent_period = 3
  AND path LIKE '%/شمال-الرياض/%'
'''

df = pd.read_sql_query(query, conn)
conn.close()

print(df.shape)

Saving archive (2).zip to archive (2).zip
(28052, 12)


In [2]:
print(df.isna().sum())
print(df.duplicated().sum())

df_clean = df.copy()
df_clean['create_time'] = pd.to_datetime(df_clean['create_time'], unit='s', errors='coerce')

rename_map = {
    'id': 'listing_id',
    'price': 'annual_rent_sar',
    'area': 'area_sqm',
    'district': 'district_name',
    'beds': 'bedrooms',
    'livings': 'living_rooms',
    'wc': 'bathrooms'
}
df_clean = df_clean.rename(columns=rename_map)

print(df_clean.head())

id               0
title            0
price            0
area            48
beds            12
livings         23
wc               3
ac              14
ketchen         31
furnished      100
district         0
create_time      0
dtype: int64
0
   listing_id                                              title  \
0      211899     شقة للإيجار في شارع الزمرد, السليمانية, الرياض   
1     1151469               شقة للإيجار في شارع العارض، ، الرياض   
2     2525972  شقة للإيجار في شارع الغطغط ، حي الربيع ، الريا...   
3     2525991  شقة للإيجار في شارع الغطغط ، حي الربيع ، الريا...   
4     3137603  شقة للإيجار في شارع محمد المقدمي ، حي الغدير ،...   

   annual_rent_sar  area_sqm  bedrooms  living_rooms  bathrooms   ac  ketchen  \
0          85000.0       0.0       2.0           1.0        2.0  0.0      1.0   
1          35000.0     112.0       3.0           1.0        2.0  1.0      1.0   
2         140000.0       NaN       2.0           1.0        2.0  1.0      1.0   
3         145000.0      

In [3]:
valid_mask = (
    (df_clean['annual_rent_sar'] >= 20000) & (df_clean['annual_rent_sar'] <= 150000) &
    (df_clean['area_sqm'] >= 40) & (df_clean['area_sqm'] <= 250) &
    (df_clean['district_name'].notna()) & (df_clean['district_name'].str.strip() != '')
)
df_clean = df_clean[valid_mask].copy()

binary_map = {0: 'لا', 1: 'نعم'}
for col in ['ac', 'ketchen', 'furnished']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(binary_map).fillna('لا')

df_clean['district_name'] = df_clean['district_name'].str.strip()

print(df_clean.shape)
print(df_clean[['annual_rent_sar', 'area_sqm']].describe())

(23190, 12)
       annual_rent_sar      area_sqm
count     23190.000000  23190.000000
mean      57166.161329    141.171238
std       24034.246668     43.516273
min       20000.000000     40.000000
25%       38000.000000    113.000000
50%       50000.000000    148.000000
75%       75000.000000    170.000000
max      150000.000000    250.000000


In [4]:
df_clean['price_per_sqm'] = (df_clean['annual_rent_sar'] / df_clean['area_sqm']).round(2)

print(df_clean[['annual_rent_sar', 'area_sqm', 'price_per_sqm']].head())

    annual_rent_sar  area_sqm  price_per_sqm
1           35000.0     112.0         312.50
4          150000.0     133.0        1127.82
8           60000.0     200.0         300.00
9           40000.0      80.0         500.00
10          31000.0      48.0         645.83


In [7]:
counts = df_clean['district_name'].value_counts()
valid_districts = counts[counts >= 10].index

df_filtered_districts = df_clean[df_clean['district_name'].isin(valid_districts)]

district_stats = df_filtered_districts.groupby('district_name')['annual_rent_sar'].agg(['mean', 'max', 'min']).sort_values(by='mean', ascending=False)
district_stats.columns = ['متوسط الإيجار السنوي', 'أعلى سعر إيجار', 'أدنى سعر إيجار']

print(f"متوسط الإيجار العام لشمال الرياض: {df_filtered_districts['annual_rent_sar'].mean():,.2f} ريال")
print("\n--- الترتيب الصحيح والواقعي للأحياء (الأحياء التي تحتوي على 10 شقق فأكثر) ---")
print(district_stats)

متوسط الإيجار العام لشمال الرياض: 57,166.18 ريال

--- الترتيب الصحيح والواقعي للأحياء (الأحياء التي تحتوي على 10 شقق فأكثر) ---
                           متوسط الإيجار السنوي  أعلى سعر إيجار  \
district_name                                                     
حي المعذر الشمالي                  90090.909091        130000.0   
حي النخيل                          87089.621348        150000.0   
حي صلاح الدين                      77255.813953        150000.0   
حي الرحمانية                       75466.666667        150000.0   
حي الندى                           74699.230769        144000.0   
حي المحمدية                        73047.619048        150000.0   
حي الواحة                          71166.666667        120000.0   
حي حطين                            69074.051724        150000.0   
حي الملقا                          67200.055102        150000.0   
حي الربيع                          63545.294118        150000.0   
حي الغدير                          62640.076235        150000.0   
ح

In [8]:
df_final = df_clean[df_clean['district_name'].isin(valid_districts)].copy()

final_output_name = "riyadh_north_apartments.csv"
df_final.to_csv(final_output_name, index=False, encoding='utf-8-sig')

from google.colab import files
files.download(final_output_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>